# 00 · Core tests (v4)

Runs the `sepsis_core` test-suite. **Run this before any training notebook; do not train until every test is green.**

| Part | Data | What it proves |
|---|---|---|
| A | Synthetic (generated here, no patient data) | The eight required tests plus end-to-end smoke training and the MLflow fix |
| B | Real MIMIC-IV **FIT/VAL only**, subject-level smoke subsample | The same suite on the real schema; cohort counts and split fingerprint match v3 |
| C | Real VAL, v3 checkpoint (optional) | The v4 core reproduces v3's VAL AUROC 0.8073 from the saved checkpoint |

Governance: every cell prints aggregates only (counts, pass/fail, maximum absolute differences). No row-level frame is displayed. LOC / CONF / EXT are never loaded here; the code refuses to.

In [ ]:
# ============================ CELL 1 · BOOTSTRAP (the only place the Drive folder is named) ============================
PROJECT_ROOT = "/content/drive/MyDrive/Sepsis Attention Research"   # <- EDIT if your project folder lives elsewhere on Drive

import os, sys
sys.dont_write_bytecode = True                                       # keep __pycache__ out of the Drive folder
if "google.colab" in sys.modules and not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_ROOT = os.environ.get("SEPSIS_PROJECT_ROOT", PROJECT_ROOT)
if not os.path.isdir(os.path.join(PROJECT_ROOT, "Notebooks", "v4", "sepsis_core")):
    raise FileNotFoundError(f"'{PROJECT_ROOT}' does not contain Notebooks/v4/sepsis_core. Is Drive mounted, and is PROJECT_ROOT (first line of this cell) "
                            "the folder that holds Notebooks/ and Docs/?")
os.environ["SEPSIS_PROJECT_ROOT"] = PROJECT_ROOT
if os.path.join(PROJECT_ROOT, "Notebooks", "v4") not in sys.path: sys.path.insert(0, os.path.join(PROJECT_ROOT, "Notebooks", "v4"))

import subprocess, importlib.util
if importlib.util.find_spec("mlflow") is None:                       # optional: only test 10 needs it
    subprocess.call([sys.executable, "-m", "pip", "install", "-q", "mlflow"])

import json, time
import numpy as np, torch
import sepsis_core
from sepsis_core import config as C, data, dataset, models, engine, metrics, synthetic, tests
print("sepsis_core", sepsis_core.__version__, "| torch", torch.__version__, "| device", engine.RT.device, "| amp", engine.RT.amp, "| grad scaler", engine.RT.scaler)
print("project root:", C.PATHS.PROJECT_ROOT)

In [ ]:
# ============================ CELL 2 · SWITCHES ============================
SMOKE_TEST   = True      # Part B uses a ~3% subject-level subsample of FIT/VAL. The tests do not need more.
RUN_REAL     = True      # False = synthetic only (Part A)
RUN_V3_REPRO = False     # Part C: score the v3 checkpoint on ALL of VAL with the v4 core (a few minutes on a GPU). Expect AUROC 0.8073.
if SMOKE_TEST: C.CFG.apply_smoke()
C.CFG.NUM_WORKERS = 0 if engine.RT.device.type == "cpu" else 2
HAVE_REAL = RUN_REAL and os.path.exists(C.PATHS.DATA_PATH)
print("smoke:", C.CFG.SMOKE_TEST, "| real data found:", HAVE_REAL, "| run name:", C.CFG.RUN_NAME)

## Part A · Synthetic data

Each line is `PASS`/`FAIL` plus the evidence. For tests 1, 4 and 6 the "must not change" quantities are expected to be exactly `0.0` (they run on CPU in fp32).

Test 6 also **reports** two numbers it does not assert (`REPORTED_layers_ge2_...`): in layers ≥ 2 the change in logits for keys *before* the intervened hour is not zero, because the query has already read that hour. See `Docs/DECISIONS.md`, open item O1.

In [ ]:
# ============================ CELL 3 · PART A: the suite on synthetic data ============================
t0 = time.time()
RES_SYN = tests.run_all()            # builds its own synthetic cohort (500 subjects) through the real preprocessing path
print(f"\nPart A took {time.time() - t0:.0f}s")

## Part B · Real data, FIT/VAL only, smoke subsample

Loads the hourly parquet, applies the v3 label fix, **loads** the frozen split (never rebuilds it) and checks its fingerprint, keeps FIT/VAL, then runs the same suite. The cohort counts are compared with v3 before subsampling.

In [ ]:
# ============================ CELL 4 · PART B: load real FIT/VAL (aggregates only) ============================
BUNDLE_REAL, DF_VAL, STATIC_REAL = None, None, None
if HAVE_REAL:
    C.require_mounted()
    df, STATIC_REAL, STATS = data.load_hourly()
    print("cohort:", json.dumps(STATS))
    V3_REF = dict(rows=3_966_669, patients=56_075, stays=74_704)
    for k, ref in V3_REF.items():
        print(f"  {k}: {STATS[k]:,} (v3: {ref:,}) {'OK' if STATS[k] == ref else '<-- DIFFERS FROM v3'}")
    print(f"  positive rate: {STATS['pos_rate']:.4%} (v3: 1.1461%)")
    ONSET = data.stay_onset(df)
    n_sept = int(ONSET.notna().sum()); n_early = int((ONSET < C.CFG.EARLY_ONSET_HOURS).sum())
    print(f"septic stays: {n_sept:,} | early-onset (< {C.CFG.EARLY_ONSET_HOURS} h): {n_early:,} ({n_early / max(1, n_sept):.1%}) (v3: 11,731 / 7,659 / 65.3%)")
    split, sha = data.load_frozen_split()
    print("frozen split sha1:", sha, "(expected", C.CFG.SPLIT_SHA1 + ")")
    df, fold_counts = data.attach_folds(df, split, ("FIT", "VAL"), data.FoldGuard("train"))
    print("subjects per fold in the split:", fold_counts, "| loaded folds:", sorted(df['fold'].unique()))
    print(f"stay length: max {int(df.groupby(C.CFG.ID_COL)[C.CFG.HOUR_COL].max().max())} h (MAX_SEQ_LEN = {C.CFG.MAX_SEQ_LEN})")
    if RUN_V3_REPRO: DF_VAL = df[df["fold"] == "VAL"].copy()
    if C.CFG.SMOKE_TEST: df = data.smoke_subsample(df)
    pre = data.Preprocessor(static_cols=STATIC_REAL).fit(df, (df["fold"] == "FIT").to_numpy(), fold=df["fold"].to_numpy())
    store = pre.transform(df)
    fit_idx = np.flatnonzero(store.stay_fold == "FIT"); val_idx = np.flatnonzero(store.stay_fold == "VAL")
    pr, alpha, pw = data.prevalence_alpha(store, fit_idx)
    print(f"stays: FIT={len(fit_idx):,} VAL={len(val_idx):,} | rows={store.n_rows:,} | hour-gap violations={store.gap_violations:,} | "
          f"FIT prevalence per horizon={[round(p, 4) for p in pr]} | focal alpha={alpha:.4f}")
    BUNDLE_REAL = dict(store=store, pre=pre, fit_idx=fit_idx, val_idx=val_idx, onset=ONSET, alpha=alpha, pw=pw, df=df)
else:
    print("Part B skipped: real data not found at", C.PATHS.DATA_PATH, "(or RUN_REAL=False).")

In [ ]:
# ============================ CELL 5 · PART B: the suite on real FIT/VAL ============================
RES_REAL = None
if BUNDLE_REAL is not None:
    t0 = time.time()
    RES_REAL = tests.run_all(BUNDLE_REAL)
    print(f"\nPart B took {time.time() - t0:.0f}s")

## Part C · Does the v4 core reproduce v3? (optional)

Loads v3's primary checkpoint and its saved FIT-only preprocessor, scores **all of VAL** with the v4 engine in v3's view (most recent 168 h per stay) and compares with v3's 0.8073. Set `RUN_V3_REPRO = True` in Cell 2 to run it.

In [ ]:
# ============================ CELL 6 · PART C: v3 checkpoint through the v4 core ============================
V3_REPRO = None
if RUN_V3_REPRO and DF_VAL is not None:
    ckpt = os.path.join(C.PATHS.V3_FINAL_DIR, "best.pt")
    if not os.path.exists(ckpt):
        print("v3 checkpoint not found at", ckpt)
    else:
        ck = torch.load(ckpt, map_location="cpu", weights_only=False)
        pre3 = data.Preprocessor.from_dict(ck["preproc"])                       # v3's own FIT-only statistics
        store3 = pre3.transform(DF_VAL)
        m3 = models.build_model(ck["hp"], pre3.x_mean); m3.load_state_dict(ck["model"]); m3.to(engine.RT.device).eval()
        loader, _ = dataset.make_loader(store3, np.arange(len(store3.starts)), C.CFG.EVAL_BATCH, False, max_len=168, mode="last")
        rows, lg, _ = engine.predict_loader(m3, loader, as_logits=True)
        met = metrics.compute_metrics(store3, rows, lg)
        V3_REPRO = dict(auroc_mean=met["auroc_mean"], auroc_t1=met["auroc_t1"], auroc_t3=met["auroc_t3"], auroc_t6=met["auroc_t6"], v3_reported=0.8073,
                        abs_diff=abs(met["auroc_mean"] - 0.8073), n_stays=int(len(store3.starts)), n_rows=int(len(rows)))
        print(json.dumps(V3_REPRO, indent=1))
        print("REPRODUCED" if V3_REPRO["abs_diff"] < 1e-3 else "<-- DIFFERS from v3 by more than 0.001: tell Claude before training")
        del m3, store3
else:
    print("Part C skipped (RUN_V3_REPRO=False or no real data).")

In [ ]:
# ============================ CELL 7 · SUMMARY + REPORT ============================
def _summ(res): return None if res is None else dict(passed=sum(r["passed"] for r in res), total=len(res), failed=[r["name"] for r in res if not r["passed"]])
REPORT = dict(time=time.strftime("%Y-%m-%dT%H:%M:%S"), sepsis_core=sepsis_core.__version__, torch=torch.__version__, device=str(engine.RT.device),
              amp=str(engine.RT.amp), synthetic=_summ(RES_SYN), real_smoke=_summ(RES_REAL), v3_repro=V3_REPRO,
              details=dict(synthetic=RES_SYN, real_smoke=RES_REAL))
out_dir = os.path.join(C.PATHS.WORK_DIR if HAVE_REAL else ".", "reports"); os.makedirs(out_dir, exist_ok=True)
out = os.path.join(out_dir, "00_core_tests.json"); json.dump(REPORT, open(out, "w"), indent=1, default=str)
print("synthetic :", REPORT["synthetic"]); print("real smoke:", REPORT["real_smoke"]); print("v3 repro  :", V3_REPRO); print("report saved:", out)
ALL_GREEN = all(s is None or s["passed"] == s["total"] for s in (REPORT["synthetic"], REPORT["real_smoke"]))
print("\nALL GREEN - safe to move to Phase 2" if ALL_GREEN else "\nNOT GREEN - send the output of this cell (aggregates only) back before training anything")